# Bulldozer Price Prediction Project

Predicting the sale price of bulldozers using machine learning.

In this notebook, we're going to go through an example machine learning
project with the goal of predicting the sale price of bulldozers.


## 1. Problem definition

> How well can we predict the future sales price of a bulldozer,
> given its characteristics and previous examples of the past
> sales price of similar bulldozers.

## 2 Data

The data is downloaded from the Kaggle Bluebook for Bulldozers competition.

From [the website](https://www.kaggle.com/c/bluebook-for-bulldozers/data):

The data for this competition is split into three parts:

- Train.csv is the training set, which contains data through the end of 2011.
- Valid.csv is the validation set, which contains data from
January 1, 2012 - April 30, 2012 You make predictions on this set
throughout the majority of the competition. Your score on this set is used
to create the public leaderboard.
- Test.csv is the test set, which won't be released until the last week
of the competition. It contains data from May 1, 2012 - November 2012.
Your score on the test set determines your final rank for the competition.

The key fields are in train.csv are:

- SalesID: the uniue identifier of the sale
- MachineID: the unique identifier of a machine.  A machine can be
sold multiple times
- saleprice: what the machine sold for at auction (only provided
in train.csv)
- saledate: the date of the sale

There are several fields towards the end of the file on the different
options a machine can have.  The descriptions all start with
"machine configuration" in the data dictionary.  Some product types
do not have a particular option, so all the records for that option
variable will be null for that product type.  Also, some sources do not
provide good option and/or hours data.

The machine_appendix.csv file contains the correct year manufactured
for a given machine along with the make, model, and product class
details. There is one machine id for every machine in all the
competition datasets (training, evaluation, etc.).


## 3. Evaluation

Again, from
[the Kaggle competition evaluation](www.kaggle.com/competitions/bluebook-for-bulldozers/overview/evaluation).

The evaluation metric for this competition is the RMSLE (room mean
squared log error) between the actual and predicted auction prices.

For more for the evaluation of this project check the Kaggle evaluation
section (see above).

**Note**: The goal for most regression evaluation metrics is to minimize
the error. For example, our goal for this project is to build a machine
learning model which minimizes RMSLE.

## 4. Features

Kaggle provides a data dictionary detailing all the features of the
data set. You can view this data dictionary using Excel, Mac Numbers,
Google Sheets, or even using PyCharm.


In [ ]:
# Import required packages

# Import cytoolz
import cytoolz.curried as ctc

# Import data analysis packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import machine learning packages
import sklearn
from matplotlib.rcsetup import validate_int_or_None

In [ ]:
# Create a `default_rng` for `numpy`
rng = np.random.default_rng(42)

In [ ]:
# Load training **and** validation sets.
# Reason will be clarified in later video.
df = pd.read_csv('data/bluebook-for-bulldozers/TrainAndValid.csv')

Daniel encountered the same warning in the video. His solution:
set `low_memory=False`.

In [ ]:
del df

In [ ]:
df = pd.read_csv(
    'data/bluebook-for-bulldozers/TrainAndValid.csv',
    low_memory=False, # Tells pandas to **not** try to minimize space
)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
# Let's plot some columns
fig, ax = plt.subplots()
ax.scatter(df['saledate'], df['SalePrice'])
plt.show()

Hmmm. Plotting all the data is **not** very useful - just as Daniel
said in the video. :)

In [ ]:
# Let's plot a subset of the data
fig, ax = plt.subplots()
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])
plt.show()

In [ ]:
df.saledate[:1000]

A scatter plot, even of only a portion of the data, does not seem
very useful either.  Let's try plotting a histogram.

In [ ]:
# Always a good idea to plot our target variable(s)
df['SalePrice'].plot.hist()
plt.show()

### Parsing dates

When we look at timeseries data, we want to "enrich" the time and date
component as much as possible.

We can accomplish this by telling `pandas` which of our columns contain
dates using the `parse_dates` parameter.

In [ ]:
# Notice the previous `dtype`
df.saledate.dtype

In [ ]:
# Import data again but this time parse dates
df = pd.read_csv(
    'data/bluebook-for-bulldozers/TrainAndValid.csv',
    low_memory=False,
    parse_dates=['saledate'],
)

In [ ]:
df.saledate.dtype

In [ ]:
df['saledate'][:1000]

In [ ]:
# Let's plot our limited scatter plot again
fig, ax = plt.subplots()
ax.scatter(df['saledate'][:1000], df['SalePrice'][:1000])
plt.show()

In [ ]:
df.head()

In [ ]:
# A trick for viewing all our columns
df.head().T

In [ ]:
# Let's remind ourselves of our sales data
df.saledate.head(20)

## Sort `DataFrame` by `saledate`

When working with time series data, it is a good practice to sort
the data by the time point of interest. (In this case, `saledate`).

In [ ]:
# Sort `DataFrame` in date order
df.sort_values(by=['saledate'], ascending=True, inplace=True)
df.saledate.head(20)

In [ ]:
df.head()

In [ ]:
# Make a copy
df_tmp = df.copy()
df_tmp

### Add datetime parameters to `saledate` column


In [ ]:
df.columns

In [ ]:
df_tmp[:1].saledate.dt.year

In [ ]:
df_tmp[:1].saledate.dt.day

In [ ]:
df_tmp[:1].saledate

In [ ]:
df_tmp['saleyear'] = df_tmp['saledate'].dt.year
df_tmp['salemonth'] = df_tmp.saledate.dt.month
df_tmp['saleday'] = df_tmp.saledate.dt.day
df_tmp['saledayofweek'] = df_tmp.saledate.dt.dayofweek
df_tmp['saledayofyear'] = df_tmp.saledate.dt.dayofyear


In [ ]:
df_tmp.head().T.tail()

In [ ]:
# Now that we've enriched our `DataFrame` with date time features,
# we can remove the `saledate` column (from `df_tmp`)
df_tmp.drop('saledate', axis=1, inplace=True)

In [ ]:
'saledate' in df_tmp.columns

In [ ]:
# Which state has the most soles?
df_tmp.state.value_counts()

In [ ]:
len(df_tmp)

## 5. Modeling

We've done enough exploratory data analysis (EDA - we could always
do more). Let's start modelling! More specifically, let's do some
_model-driven EDA_.

What kind of learning should we do?

Back to our [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html).

Walking through the map and primarily because we have experience
with this learning, lets build a `RandomForestRegressor`

In [ ]:
# Let's build a machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_jobs=-1, # use all available processors
    random_state=rng.integers(np.iinfo(np.int32).max),
)

# Our target is 'SalePrice' so we drop it from the data
# **before** fitting and supply that column as the target
# column for fitting our model.
#
# In our X and y nomenclature, X is all the columns but `SalePrice`
# and y is the `SalePrice` column.
try:
    model.fit(df_tmp.drop('SalePrice', axis=1), df_tmp['SalePrice'])
except ValueError as ve:
    print(ve)

In [ ]:
df.info()

In [ ]:
# Let's look at one of the columns of type `object`
df_tmp['UsageBand'].head()

### Convert strings to categories

One way to turn all of data into numbers is to convert them to pandas
`Categorical` instances.

Pandas `Categorical` types

- [Categorical API](https://pandas.pydata.org/docs/reference/arrays.html#categoricals
- [Categorical data user guide](https://pandas.pydata.org/docs/user_guide/categorical.html)

Additionally, see [Pandas arrays, scalars and data types](https://pandas.pydata.org/docs/reference/arrays.html#pandas-arrays-scalars-and-data-types)

Machine learning is about

- Getting data
- "Massaging" the data into a format amenable to a machine learning moel
- Find patterns
- Evaluating how well our steps worked


In [ ]:
df_tmp.head().T

In [ ]:
# Let's investigate
pd.api.types.is_string_dtype(df_tmp['UsageBand'])

In [ ]:
df_tmp['UsageBand']

In [ ]:
df_tmp[df_tmp['UsageBand'].notnull()]['UsageBand']

Let's investigate the `UsageBand` column
- `pd.api.types.is_string_dtype(df_tmp['UsageBand'])`
- Hmm...
	- In the video, this function returned `True`
	- However, in my environment, this function returns `False`

This text summarizes the changes to pandas function `is_string_dtype()`.
It is copied from the experimental AI Overview generated on
2024-01-28.

The behavior of pandas.api.types.is_string_dtype has changed in recent
versions of pandas, particularly with the introduction of the dedicated
string dtype in pandas 2.0 and its enhancements in pandas 2.2.

Here's a breakdown of the changes:

**Pandas 2.0 and earlier**:

- `is_string_dtype` would return `True` for object dtype columns if
all the elements in the column were strings.
- It would return `False` for columns with mixed data types or
non-string objects.

**Pandas 2.2 and later:**

- Pandas introduced a new string dtype (`string`) backed by Arrow,
which provides significant performance and memory improvements.
- `is_string_dtype` now returns `True` for columns with the new string dtype.
- It still returns `True` for object dtype columns if all elements
are strings, but this behavior is considered deprecated.

**Pandas 3.0 and later:**

- The default string dtype will be the Arrow-backed string dtype,
and `is_string_dtype` will primarily be used to identify columns with
this dtype.
- PyArrow will become a required dependency for pandas.

**Key points to remember:**

- If you're using pandas 2.2 or later, consider enabling the new
string dtype inference with `pd.options.future.infer_string = True`.
- If you're relying on `is_string_dtype` to check for object `dtype`
columns containing only strings, be aware that this behavior is deprecated
and may change in the future.
- For better performance and memory efficiency, use the new string dtype when working with string data in pandas.

Generative AI is experimental.

In [ ]:
# find the columns which contain strings
# for label, content in df_tmp.items():
#     if pd.api.types.is_string_dtype(content):
#         print(label)
for label in df_tmp.columns:
    if df_tmp[label].dtype == 'object':
        try:
            df_tmp[label].astype(str)
            print(label)
        except ValueError:
            print(f'Cannot convert column, {label}, to string.')

In [ ]:
# If you wondering what df.items() does, here's an example.
random_dict = {'key1': 'hello',
               'key2': 'world!'}

for key, value in random_dict.items():
    print(f'this is a key: {key}',
          f', this is a value: {value}')

In [ ]:
# I think I can make this code work, but a bit too much time right now.
# ctc.pipe(
#     df_tmp.columns,
#     ctc.filter(lambda cn: df_tmp[cn].dtype == 'object'),
#     ctc.map(lambda cn: df_tmp[cn].map(str)),
#     ctc.filter(lambda cn: cn == 'UsageBand'),
#     pd.Series,
# )

In [ ]:
columns_to_convert = [label for
                      label in df_tmp.columns
                      if df_tmp[label].dtype == 'object']
for label in columns_to_convert:
    df_tmp[label] = df_tmp[label].astype('string')

In [ ]:
df_tmp[df_tmp['UsageBand'].notnull()]['UsageBand']

In [ ]:
# And now back to our regularly scheduled video. :)
# This will turn all of the string values into category values
for label, content in df_tmp.items():
    if pd.api.types.is_string_dtype(content):
        df_tmp[label] = content.astype('category').cat.as_ordered()

In [ ]:
df_tmp.info()

In [ ]:
df_tmp.state.cat.categories

In [ ]:
df_tmp.state.cat.codes

Thanks to pandas `Categorical` types, we now a way to access all our
data in the form of numbers.

However, we're still **missing some data**.

In [ ]:
# Fraction of missing data for each column
df_tmp.isnull().sum() / len(df_tmp)

In [ ]:
# Export current tmp dataframe (a checkpoint of our work)
df_tmp.to_csv('./data/bluebook-for-bulldozers/train_tmp.csv', index=False)

In [ ]:
# And import the preprocessed data
df_tmp = pd.read_csv('./data/bluebook-for-bulldozers/train_tmp.csv',
                     low_memory=False)
df_tmp.head().T

In [ ]:
# Remind ourselves that we still have **missing** values
df_tmp.isna().sum()

## Fill missing values

### Fill numeric missing values first

In [ ]:
# Find out which columns are numeric
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        print(label)

In [ ]:
numeric_columns = ctc.pipe(
    df_tmp.items(),
    ctc.filter(lambda t: pd.api.types.is_numeric_dtype(t[1])),
    ctc.map(ctc.first),
    list,
)
numeric_columns

In [ ]:
df_tmp['ModelID']

In [ ]:
# Check for which numeric columns hane null values
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# Or...
ctc.pipe(
    df_tmp.items(),
    ctc.filter(lambda t: pd.api.types.is_numeric_dtype(t[1])),
    ctc.filter(lambda t: pd.isnull(t[1]).sum()),
    ctc.map(ctc.first),
    list,
)

In [ ]:
# Fill the null values in the (two) numeric columns
# with the median of the column.
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            # Add a binary column which tells us if data is missing
            # This action allows us to **remember** if we filled
            # missing values with the median
            df_tmp[label + '_is_missing'] = pd.isnull(content)

            # Fill missing numeric values with median
            # Using median because median is more robust
            # against outliers.
            df_tmp[label] = content.fillna(content.median())

In [ ]:
# Demonstrate that the median is more robust that the mean
hundreds = np.full((1000,), 100)
hundreds_billion = np.append(hundreds, 1_000_000_000)
np.mean(hundreds), np.mean(hundreds_billion), np.median(hundreds), np.median(hundreds_billion)

In [ ]:
# Check again for null numeric values
for label, content in df_tmp.items():
    if pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# But how many **were missing**?
df_tmp['auctioneerID_is_missing'].value_counts()

In [ ]:
df_tmp.columns

In [ ]:
# Remember all our columns with missing data
df_tmp.isna().sum()

### Filling and turning `Categorical` variables into numbers

In [ ]:
# Check for columns that are **not** numeric
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        if pd.isnull(content).sum():
            print(label)

In [ ]:
# Let's examine a column of type `Categorical`
pd.Categorical(df_tmp['state'])

In [ ]:
pd.Categorical(df_tmp.state).dtype

In [ ]:
pd.Categorical(df_tmp['state']).codes

In [ ]:
pd.Categorical(df_tmp.state).codes == -1

In [ ]:
pd.Categorical(df_tmp['UsageBand']).codes

In [ ]:
pd.Categorical(df_tmp['UsageBand']).codes == -1

In [ ]:
np.sum(pd.Categorical(df_tmp['UsageBand']).codes != -1)

In [ ]:
# Turn categorical values into numbers and fill missing
for label, content in df_tmp.items():
    if not pd.api.types.is_numeric_dtype(content):
        # Add a binary column to indicate missing data
        df_tmp[label + '_is_missing'] = pd.isnull(content)

        # Turn categories into numbers **but add 1**
        # Add one because the `Categorical` dtype encodes missing values
        # as **-1**. Adding 1 means that all our values are non-negative.
        df_tmp[label] = pd.Categorical(content).codes + 1

In [ ]:
# Let's see what we've done
df_tmp.info()

In [ ]:
df_tmp.head().T

In [ ]:
# Let's ensure we have no missing values
np.any(df_tmp.isna().sum() != 0)

Now that all of our data is numeric and we have no missing values,
we should be able to build a machine learning model.

In [ ]:
df_tmp.head()

In [ ]:
len(df_tmp)

In [ ]:
%%time
# Instantiate model
model = RandomForestRegressor(n_jobs=-1,
                              random_state=rng.integers(np.iinfo(np.int32).max,))

# Fit the model
model.fit(df_tmp.drop('SalePrice', axis=1), df_tmp['SalePrice'])

In [ ]:
# Score the model
model.score(df_tmp.drop('SalePrice', axis=1), df_tmp['SalePrice'])

g**Question**: Why does the above metric "hold water"?

In other words, why is the metric (score) not reliable?

Because we

- Trained our data on some data
- But evaluated it on **the same data**

## Splitting the data into training and validation sets

In [ ]:
df_tmp.head()

In [ ]:
# Can we use the year to split our data?
df_tmp.saleyear

In [ ]:
df_tmp.columns

In [ ]:
list(ctc.filter(lambda cn: cn.startswith('sale'), df_tmp.columns))

In [ ]:
df_tmp.saleyear.value_counts()

In [ ]:
# Split data into training and validation
# df_val = df_tmp[df_tmp['saleyear' == 2012]]
# df_tmp[df_tmp['saleyear' == 2012]].value_counts()
# (df_tmp.saleyear == 2012).sum()
df_valid = df_tmp[df_tmp.saleyear == 2012]
df_train = df_tmp[df_tmp.saleyear != 2012]
len(df_valid), len(df_train)

In [ ]:
# Split data in X and y (features and labels)
X_train, y_train, = df_train.drop('SalePrice', axis=1), df_train['SalePrice']
X_valid, y_valid = df_valid.drop('SalePrice', axis=1), df_valid['SalePrice']

# Check shapes to ensure we've not made an error
X_train.shape, y_train.shape, X_valid.shape, y_valid.shape

### Building an evaluation function

In [ ]:
# Create an evaluation function
# We will use this function many times during our experiments.
# The competition uses RMSLE. In the video, this metric **is not**
# available in `sklearn`; however, this metric is available in the
# version that I am using. I will use it to check my work.
from sklearn.metrics import (
    mean_squared_log_error,
    root_mean_squared_log_error,
    mean_absolute_error,
    r2_score,
)

def rmsle(y_test, y_pred):
    """
    Calculates the root mean squared log error between predictions and
    true values.
    :param y_test: The true values
    :param y_pred: The predicted values
    :return: The root mean squared log error
    """
    return np.sqrt(mean_squared_log_error(y_test, y_pred))

# In addition, let's create function to calculate a number of metrics
# for easy comparison
def show_scores(mode0):
    train_predictions = model.predict(X_train)
    print(len(train_predictions))
    validation_predictions = model.predict(X_valid)
    print(len(validation_predictions))

    scores = {'Training MAE': mean_absolute_error(y_train, train_predictions),
              'Validation MAE': mean_absolute_error(y_valid, validation_predictions),
              'Training custom RMSLE': rmsle(y_train, train_predictions),
              'Validation custom RMSLE': rmsle(y_valid, validation_predictions),
              'Training RMSLE': root_mean_squared_log_error(y_train, train_predictions),
              'Validation RMSLE': root_mean_squared_log_error(y_valid, validation_predictions),
              'Training R^2': r2_score(y_train, train_predictions),
              'Validation R^2': r2_score(y_valid, validation_predictions),
              }
    return scores

### Testing our model on a subset

...to tune the hyperparameters

Previous testing of the complete model took over 5 minutes in the
video (but only 1 minute on my system). When running experiments,
taking 5 minutes per test run is too long to be effective. To lessen
the time taken in testing, we can evaluate our model on a **subset**
of our data instead of testing on the entire model.

In [ ]:
# This action takes far too long (in the video) for experimenting
# %% time
# model = RandomForestRegressor(n_jobs=-1, random_state=42)
# model.fit(X_train, y_train)

In [ ]:
len(X_train)

In [ ]:
X_train.shape[0] * 100

In [ ]:
# Change the `max_samples` vaule
model = RandomForestRegressor(
    n_jobs=-1,
    random_state=42,
    max_samples=10_000,
)

In [ ]:
%%time
# Cutting down on the number of samples each estimator sees will
# improve the training time.
model.fit(X_train, y_train)

In [ ]:
show_scores(model)

In [ ]:
len(X_train), len(y_train)

### Hyper-parameter tuning with `RandomizedSearchCV`

In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV

# Different `RandomForestRegressor` hyperparameters
rf_grid = {
    'n_estimators': np.arange(10, 100, 10),
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': np.arange(2, 20, 2),
    'min_samples_leaf': np.arange(1, 20, 2),
    'max_features': [0.5, 1, 'sqrt', 'log2'],
    'max_samples': [10000],
}

# Instantiate `RandomizedSearchCV` model
rs_model = RandomizedSearchCV(
    estimator=RandomForestRegressor(n_jobs=1,
                                    random_state=rng.integers(np.iinfo(np.int32).max),),
    param_distributions=rf_grid,
    n_iter=6, # reduced to fit into video
    # Can try adjusting `n_iter` to  get better performance
    cv=5,
    verbose=True
)

# Fit the `RandomizedSearchCV` model
rs_model.fit(X_train, y_train)

In [ ]:
rs_model.best_params_

In [ ]:
# Repeat our previous `model` scores
show_scores(model)

In [ ]:
# Evaluate the model recommended by `RandomizedSearchCV
show_scores(rs_model)

Daniel ran a 6-hour tuning run. Here's what he found (and what I'll
use also):

```python
{
    'Training MAE': 6633.714300615716,
    'Valid MAE': 8074.634589086979,
    'Training RMSLE': 0.29670133662928448,
    'Valid RMSLE': 0.32142476459239144,
    'Training R^2': 0.807156908286598,
    'Valid R^2': 0.7527628271827518,
}
```

## Train a model with the "best" hyperparameters

**Note**: These parameters were after 100 iterations of
`RandomizedSearchCV`.

In [ ]:
%%time

# Most idea hyperparameters
ideal_model = RandomForestRegressor(
    n_jobs=-1,
    random_state=42, # Set `random_state` for reproducible results
    n_estimators=40,
    min_samples_leaf=1,
    min_samples_split=14,
    max_features=0.5,
    max_samples=None, # Uses **all** the samples available
)

# Fit the ideal model
ideal_model.fit(X_train, y_train)

In [ ]:
# Shows for `ideal_model` (trained on all data)
show_scores(ideal_model)

In [ ]:
# Scores on `rs_model` (only trained on ~10,000 examples)
show_scores(rs_model)

### Make predictions on test data

In [ ]:
# Import test data
df_test = pd.read_csv('data/bluebook-for-bulldozers/Test.csv',
                      low_memory=False,
                      parse_dates=['saledate'])
df_test.head()

In [ ]:
# Make predictions on the test data set
try:
    test_preds = ideal_model.predict(df_test)
except ValueError as ve:
    print(ve)

In [ ]:
# Imported data has missing values
df_test.isna().sum()

In [ ]:
# Imported data is **not** all numeric
df_test.info()

In [ ]:
df_test.columns

In [ ]:
X_train.columns

### Preprocessing the data

Getting the test data set in the same format as our training and
validation data set

In [ ]:
# Define a function to perform our preprocessing
def preprocess_dataset(df):
    # Additional columns for date "pieces"
    df['saleyear'] = df['saledate'].dt.year
    df['salemonth'] = df['saledate'].dt.month
    df['saleday'] = df['saledate'].dt.day
    df['saledayofweek'] = df['saledate'].dt.dayofweek
    df['saledayofyear'] = df['saledate'].dt.dayofyear

    # Drop the 'label' column
    df.drop('saledate', axis=1, inplace=True)

    # Not in video but needed for my version of packages
    # Converts `object` columns to `string` columns
    columns_to_convert = [
        label for label in df.columns if df[label].dtype == 'object'
    ]
    for label in columns_to_convert:
        df[label] = df[label].astype('string')

    # Now, convert all our 'string' columns to category columns
    for label, content in df.items():
       if pd.api.types.is_string_dtype(content):
           df[label] = df[label].astype('category').cat.as_ordered()

    # Fill missing values in numeric columns with median value
    for label, content in df.items():
        if pd.api.types.is_numeric_dtype(content):
            if pd.isnull(content).sum():
                # Add a binary column which tells us if the data was
                # originally missing. This action allows us to
                # **remember** if we filled missing values with the
                # median value
                df[label + '_is_missing'] = pd.isnull(content)

                # Fill missing numeric values with the median value
                # Using median because median is more robust against
                # outliers than mean
                df[label] = content.fillna(content.median())

    # Turn categorical values into numbers and fill missing values
    for label, content in df.items():
        if not pd.api.types.is_numeric_dtype(content):
            # Add binary column to indicate missing data
            df[label + '_is_missing'] = pd.isnull(content)

            # Turn categories into numbers **but add 1**
            # Add one because the `Categorical` dtype encodes
            # missing values as **-1**. Adding 1 mens that
            # all our values are non-negative.
            df[label] = pd.Categorical(content).codes + 1

    return df

In [ ]:
df_test = preprocess_dataset(df_test)
df_test.head()

In [ ]:
try:
    test_preds = ideal_model.predict(df_test)
except ValueError as ve:
    print(ve)

In [ ]:
# Additionally, we can find how columns differ by using sets
set(X_train.columns) - set(df_test.columns)

In [ ]:
df_test['auctioneerID'].isna().sum() == 0

In [ ]:
# Manually adjust `df_test` to have `auctioneerID_is_missing` column
# with all values `False`
df_test['auctioneerID_is_missing'] = False
df_test.head()

In [ ]:
set(X_train.columns) - set(df_test.columns)

Finally, our test data frame now has the same features as the training data.
So we can make predictions!

In [ ]:
# Make predictions on the test data
try:
    test_preds = ideal_model.predict(df_test)
except ValueError as ve:
    print(ve)

In [ ]:
_, df_tmp = df_test.align(X_train, join='outer', axis=1)

In [ ]:
set(X_train.columns) - set(df_tmp.columns)

In [ ]:
list(X_train.columns) == list(df_tmp.columns)

In [ ]:
try:
    test_preds = ideal_model.predict(df_tmp)
except ValueError as ve:
    print(ve)

In [ ]:
_, df_tmp = df_test.align(X_valid, axis=1)

In [ ]:
set(X_train.columns) - set(df_tmp.columns)

In [ ]:
list(X_train.columns) == list(df_tmp.columns)

In [ ]:
X_train.columns[55:65]

In [ ]:
df_tmp.columns[55:65]

In [ ]:
df_tmp_aligned = df_tmp.reindex(columns=X_train.columns)

In [ ]:
df_tmp_aligned.columns[55:65]

In [ ]:
df_test = df_test.reindex(columns=X_train.columns)

In [ ]:
set(X_train.columns) - set(df_test.columns)

In [ ]:
list(X_train.columns) == list(df_test.columns)

In [ ]:
test_preds = ideal_model.predict(df_test)
test_preds

Our predictions are **not** formatted in the same format required by
the Kaggle competition. See
www.kaggle.com/competitions/bluebook-for-bulldozers/overview/evaluation.

In [ ]:
# Format predictions into required Kaggle format
df_preds = pd.DataFrame()
df_preds['SalesID'] = df_test['SalesID']
df_preds['SalePrice'] = test_preds
df_preds

In [ ]:
# Export prediction data
df_preds.to_csv(
    'data/bluebook-for-bulldozers/my_test_predictions.csv',
    index=False
)

### Feature Importance

Feature importance seeks to determine which attributes of the data
were **most important** in predicting the **target variable**? (In
our experiment, the target variable was `SalePrice`.)

In [ ]:
# Find feature importance of our best model
ideal_model.feature_importances_

In [ ]:
len(ideal_model.feature_importances_)

In [ ]:
# Helper function for plotting feature importance
def plot_features(columns, importances, n = 20):
    """
    Plot the `n` most important features in a bar chart
    :param columns: The columns containing the important features
    :param importances: The importance of each feature
    :param n: Plot the n most important features
    :return: None
    """
    df = (pd.DataFrame({'features': columns, 'feature_importances': importances})
          .sort_values('feature_importances', ascending=False)
          .reset_index(drop=True))

    # Plot the `DataFrame`
    fig, ax = plt.subplots()
    # Only plot first `n` features
    ax.barh(df['features'][:n], df['feature_importances'][:n])
    ax.set_ylabel('Feature')
    ax.set_xlabel('Feature importance')
    ax.invert_yaxis()

    plt.show()

In [ ]:
plot_features(X_train.columns, ideal_model.feature_importances_)

In [ ]:
X_train.head()

In [ ]:
X_train['ProductSize'].value_counts()

In [ ]:
df['ProductSize'].value_counts()

In [ ]:
df['Enclosure'].value_counts()

**Question to finish**

- Why might knowing the feature importances of a trained machine learning
model be helpful?

**Final challenge**

- What other machine learning models could you try on our dataset?
(Hint: see
[the machine learning map](https://scikit-learn.org/stable/machine_learning_map.html)).
- Try to look at something like 'CatBoost' or 'XGBoost'

Remember, we are **experimenting**.

Some possible resources:

- [CatBoost](https://medium.com/@pwrxndr/catboost-classifier-a-simple-guide-for-everyone-48a2e3897251)
- [CatBoost for Regression](https://towardsdatascience.com/catboost-regression-in-6-minutes-3487f3e5b329/)
- [XGBoost](https://medium.com/@bravinwasike18/dive-into-xgboost-and-scikit-learnmachine-learning-with-xgboost-and-scikit-learn-17e2cf54f3a3)
- [XGBoost for Regression](https://machinelearningmastery.com/xgboost-for-regression/)
